# 🤖 Notebook 03 · Machine learning for asset health

Practice for **Part 3** of your visual guide (cards 3.1 → 3.17).

The setup cell creates:

| name | what it is |
|---|---|
| `fleet` | 3,000 transformers: `age_years`, `avg_load_pct`, `last_inspection_score` (some missing), `region`, `manufacturer`, `failed` (≈15% True) |
| `load_ts` | one year of **hourly** grid load (`load`, `temp`) with a DatetimeIndex |
| `ts_X`, `ts_y` | ready-made, time-ordered features/target from `load_ts` (for exercise 3.6) |
| `sensors` | 1,001 readings of `temp_c`, `load_kw`, with one planted anomaly in the last row |
| `x_over`, `y_over` | a small noisy regression problem (for overfitting, 3.10) |
| `points` | 60 2-D points in 3 clusters (for k-means, 3.13) |
| `X_lin`, `y_lin` | data for a from-scratch linear regression (true intercept 3, slopes 2 and −1) |
| `group_a`, `group_b` | two samples for a bootstrap comparison |

Exercises build on each other (3.1 → 3.5), so do them in order. Some cells take a few seconds to train.

> 💾 **Opened this from GitHub?** Colab won't keep your edits until you choose **File → Save a copy in Drive**. Do that first.

In [ ]:
#@title ▶ Run me first: loads the practice data and the checker { display-mode: "form" }
import math, random, statistics, inspect, functools, csv, io, os, warnings
from collections import Counter, defaultdict, deque
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
_G = globals()
TESTS, DONE = {}, set()

def need(name):
    if name not in _G:
        raise NameError(f"I can't find `{name}`. Run your code cell first, and use exactly that name.")
    return _G[name]

def value(name):
    v = need(name)
    if v is Ellipsis:
        raise NotImplementedError
    return v

def close(a, b, tol=1e-6):
    if a is None:
        raise AssertionError("a result is None. Did you forget `return`?")
    return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))

def expect(fn_name, args, expected, tol=None, **kw):
    f = need(fn_name)
    got = f(*args, **kw)
    shown = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()])
    if tol is not None:
        ok = got is not None and close(got, expected, tol)
    else:
        ok = got == expected
    hint = "  (Did you forget `return`?)" if got is None and expected is not None else ""
    assert ok, f"{fn_name}({shown}) should return {expected!r}, but returned {got!r}.{hint}"
    return got

def check(qid):
    try:
        TESTS[qid]()
    except NotImplementedError:
        print(f"✏️  Exercise {qid}: replace the ... with your code, run that cell, then run this check again.")
    except AssertionError as e:
        print(f"❌ Exercise {qid}: not yet. {e}")
    except NameError as e:
        print(f"⚠️  Exercise {qid}: {e}")
    except Exception as e:
        print(f"💥 Exercise {qid}: your code raised {type(e).__name__}: {e}\n   Read the message, fix your cell, re-run it, then check again. (Seeing this before you've written anything is normal.)")
    else:
        DONE.add(qid)
        print(f"✅ Exercise {qid}: all checks passed!")

def progress():
    def key(k):
        return [float(p) if p.replace('.', '', 1).isdigit() else 999 for p in k.lstrip('M').split('.')]
    keys = sorted(TESTS, key=key)
    print(f"{len(DONE)}/{len(TESTS)} exercises passed")
    print("  ".join(("✅ " if k in DONE else "⬜ ") + k for k in keys))

# ---------- practice data ----------
from sklearn.model_selection import train_test_split, TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, IsolationForest, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             mean_absolute_error, precision_recall_curve)

_rng = np.random.default_rng(42)
_n = 3000
fleet = pd.DataFrame({
    "age_years": _rng.uniform(1, 45, _n).round(1),
    "avg_load_pct": _rng.uniform(30, 110, _n).round(1),
    "last_inspection_score": _rng.integers(1, 6, _n).astype(float),
    "region": _rng.choice(["north", "south", "coast"], _n),
    "manufacturer": _rng.choice(["A", "B", "C"], _n),
})
fleet.loc[_rng.choice(_n, 100, replace=False), "last_inspection_score"] = np.nan
_logit = -6 + 0.07 * fleet["age_years"] + 0.03 * fleet["avg_load_pct"] + 0.6 * (fleet["region"] == "coast")
fleet["failed"] = _rng.random(_n) < 1 / (1 + np.exp(-_logit))

_hours = pd.date_range("2025-01-01", periods=24 * 365, freq="h")
_t = np.arange(len(_hours))
_temp = 15 + 10 * np.sin(2 * np.pi * _t / (24 * 365)) + _rng.normal(0, 2, len(_t))
_load = (500 + 80 * np.sin(2 * np.pi * (_hours.hour - 6) / 24) + 4 * np.abs(_temp - 18)
         - 40 * (_hours.dayofweek >= 5) + _rng.normal(0, 15, len(_t)))
load_ts = pd.DataFrame({"load": _load, "temp": _temp}, index=_hours)
_f = load_ts.copy()
_f["hour"], _f["dow"] = _f.index.hour, _f.index.dayofweek
for _lag in [24, 168]:
    _f[f"lag_{_lag}"] = _f["load"].shift(_lag)
_f = _f.dropna()
ts_X, ts_y = _f.drop(columns="load"), _f["load"]

sensors = pd.DataFrame({"temp_c": _rng.normal(60, 2, 1000), "load_kw": _rng.normal(400, 18, 1000)})
sensors.loc[1000] = [74.0, 470.0]

x_over = _rng.uniform(0, 10, 300).reshape(-1, 1)
y_over = np.sin(x_over.ravel()) * 10 + _rng.normal(0, 2, 300)

points = np.vstack([_rng.normal(c, 0.45, (20, 2)) for c in ([2, 2], [6, 5], [2, 7])])

X_lin = _rng.normal(size=(500, 2))
y_lin = 3 + X_lin @ np.array([2.0, -1.0]) + _rng.normal(0, 0.1, 500)

group_a = _rng.normal(100, 15, 200)
group_b = _rng.normal(95, 15, 200)

# ---------- checks (no peeking needed: solutions are below each exercise) ----------
def _test_3_1():
    Xv, yt, ye = value("X"), value("y_tr"), value("y_te")
    assert "failed" not in Xv.columns, "X must not contain the target column"
    assert len(ye) == 750 and len(yt) == 2250, f"expected 2250 train / 750 test rows, got {len(yt)} / {len(ye)}"
    assert abs(yt.mean() - ye.mean()) < 0.01, "failure rates in train and test differ: did you use stratify=y?"
TESTS['3.1'] = _test_3_1

def _test_3_2():
    m, p = value("model"), value("proba")
    assert isinstance(m, Pipeline), "model should be a sklearn Pipeline"
    assert any(isinstance(s, ColumnTransformer) for _, s in m.steps), "the first step should be a ColumnTransformer"
    assert len(p) == 750 and p.min() >= 0 and p.max() <= 1, "proba should be 750 probabilities: predict_proba(X_te)[:, 1]"
    auc = roc_auc_score(value("y_te"), p)
    assert auc > 0.75, f"ROC AUC is {auc:.3f}; expected above 0.75. Check the columns and that you fit on X_tr, y_tr."
TESTS['3.2'] = _test_3_2

def _test_3_3():
    yt, p = value("y_te"), value("proba")
    assert close(value("base_rate"), yt.mean()), "base_rate = y_te.mean()"
    assert close(value("roc"), roc_auc_score(yt, p)), "roc = roc_auc_score(y_te, proba)"
    assert close(value("pr_auc"), average_precision_score(yt, p)), "pr_auc = average_precision_score(y_te, proba)"
    assert close(value("precision"), precision_score(yt, p >= 0.5)), "precision for threshold 0.5 looks wrong"
    assert close(value("recall"), recall_score(yt, p >= 0.5)), "recall for threshold 0.5 looks wrong"
    assert close(value("accuracy_lazy"), 1 - yt.mean()), "the lazy model is right for every non-failure: accuracy = 1 - base_rate"
    print(f"   → lazy accuracy {accuracy_lazy:.0%} vs your recall {recall:.0%}: that's why accuracy misleads.")
TESTS['3.3'] = _test_3_3

def _test_3_4():
    f = need("precision_at_k")
    assert close(f([1, 0, 1, 0], [0.9, 0.8, 0.7, 0.1], 2), 0.5), "precision_at_k([1,0,1,0], [0.9,0.8,0.7,0.1], 2) should be 0.5"
    assert close(f([1, 0, 1, 0], [0.9, 0.8, 0.7, 0.1], 3), 2 / 3), "precision_at_k(..., k=3) should be 2/3"
    yt, p = value("y_te"), value("proba")
    assert close(value("p_at_50"), f(yt, p, 50)), "p_at_50 should be precision_at_k(y_te, proba, 50)"
    t = value("thr_80")
    assert recall_score(yt, p >= t) >= 0.8 - 1e-9, f"with threshold {t:.3f} recall drops below 0.8"
    pr, rc, th = precision_recall_curve(yt, p)
    assert close(t, th[rc[:-1] >= 0.8].max()), "pick the HIGHEST threshold that still gives recall ≥ 0.8"
TESTS['3.4'] = _test_3_4

def _test_3_5():
    imp = value("importances")
    assert isinstance(imp, pd.Series) and len(imp) == 5, "importances should be a Series with one value per column of X_te"
    assert list(imp.values) == sorted(imp.values, reverse=True), "sort importances from largest to smallest"
    assert value("top_feature") in ("age_years", "avg_load_pct"), f"the data was generated mostly from age and load; got {top_feature!r}"
TESTS['3.5'] = _test_3_5

def _test_3_6():
    c = np.asarray(value("cv_mae"))
    assert c.shape == (5,), "cv_mae should hold 5 numbers (one per fold)"
    assert (c > 0).all(), "flip the sign: sklearn returns negative errors"
    ref = -cross_val_score(Ridge(), ts_X, ts_y, cv=TimeSeriesSplit(n_splits=5), scoring="neg_mean_absolute_error")
    assert np.allclose(c, ref), "use Ridge(), TimeSeriesSplit(n_splits=5) on ts_X, ts_y"
    assert value("kfold_ok_for_time_series") is False, "shuffled KFold lets the model see the future: not OK"
TESTS['3.6'] = _test_3_6

def _test_3_7():
    fe = value("feat")
    for col in ["hour", "dow", "month", "lag_24", "lag_168", "roll_24"]:
        assert col in fe.columns, f"make_features should add a column {col}"
    assert fe.isna().sum().sum() == 0, "drop rows with NaN at the end of make_features"
    assert close(fe["lag_24"].iloc[0], load_ts["load"].loc[fe.index[0] - pd.Timedelta("24h")]), "lag_24 should be the load 24 hours earlier (shift(24))"
    mm, mn = value("mae_model"), value("mae_naive")
    ref_naive = mean_absolute_error(fe["load"].iloc[-720:], fe["lag_168"].iloc[-720:])
    assert close(mn, ref_naive, 1e-3), f"mae_naive should be the MAE of lag_168 over the last 720 hours ({ref_naive:.2f})"
    assert mm < mn, f"your model ({mm:.1f}) should beat the naive baseline ({mn:.1f})"
    print(f"   → {1 - mm / mn:.0%} better than “same hour last week”.")
TESTS['3.7'] = _test_3_7

def _test_3_8():
    fl = np.asarray(value("flags"))
    assert fl.dtype == bool and len(fl) == len(sensors), "flags should be a True/False array: iso.predict(sensors) == -1"
    assert fl[-1], "the planted anomaly (last row) should be flagged"
    assert 5 <= value("n_flagged") <= 20, f"with contamination=0.01 about 10 rows should be flagged, got {n_flagged}"
TESTS['3.8'] = _test_3_8

def _test_3_9():
    g, bp = value("grid"), value("best_params")
    assert hasattr(g, "best_params_"), "call grid.fit(X_tr, y_tr)"
    assert set(bp) == {"clf__n_estimators", "clf__max_depth"}, f"best_params keys should be clf__n_estimators and clf__max_depth, got {list(bp)}"
    assert len(g.cv_results_["params"]) == 4, "the grid has 2 × 2 = 4 combinations"
TESTS['3.9'] = _test_3_9

def _test_3_10():
    r = value("results")
    assert {"depth", "train_mae", "test_mae"} <= set(r.columns) and len(r) == 5, "results needs 5 rows with depth, train_mae, test_mae"
    last = r.iloc[-1]
    assert last["train_mae"] < 0.01 and last["test_mae"] > 1, "the unlimited tree should be ~0 on train but clearly worse on test: overfitting"
    bd = value("best_depth")
    exp = r.loc[r["test_mae"].idxmin(), "depth"]
    assert (bd == exp) or (pd.isna(bd) and pd.isna(exp)), f"best_depth should be the depth with the lowest test_mae ({exp})"
TESTS['3.10'] = _test_3_10

def _test_3_11():
    w = np.asarray(need("fit_linear_gd")(X_lin, y_lin))
    ref = np.linalg.lstsq(np.c_[np.ones(len(X_lin)), X_lin], y_lin, rcond=None)[0]
    assert w.shape == (3,) and np.allclose(w, ref, atol=0.02), f"weights should be close to {ref.round(2)}, got {np.round(w, 2)}"
    cm = need("confusion_metrics")
    p, r, f = cm([1, 0, 1, 1, 0], [1, 1, 0, 1, 0])
    assert close(p, 2 / 3) and close(r, 2 / 3) and close(f, 2 / 3), f"expected (0.667, 0.667, 0.667), got {(p, r, f)}"
    assert tuple(cm([0, 0], [0, 0])) == (0.0, 0.0, 0.0), "no positives at all should give (0.0, 0.0, 0.0), not a division error"
TESTS['3.11'] = _test_3_11

def _test_3_12():
    lr = value("logreg")
    ref = lr.predict_proba(pd.DataFrame({"age_years": [30]}))[0, 1]
    assert close(value("p_at_30"), ref, 1e-6), f"p_at_30 should be {ref:.3f}: predict_proba(...)[0, 1]"
    assert close(value("odds_per_year"), np.exp(lr.coef_[0, 0])), "odds_per_year = np.exp(logreg.coef_[0, 0])"
    assert odds_per_year > 1, "older transformers fail more, so the odds multiplier should be above 1"
TESTS['3.12'] = _test_3_12

def _test_3_13():
    assert list(value("sizes")) == [20, 20, 20], f"each of the 3 clusters should have 20 points, got {sizes}"
    assert value("km").cluster_centers_.shape == (3, 2), "km should be fitted with 3 clusters"
TESTS['3.13'] = _test_3_13

def _test_3_14():
    s = np.asarray(value("scaled"))
    assert s.shape == (100, 2) and np.allclose(s.mean(axis=0), 0, atol=1e-9) and np.allclose(s.std(axis=0), 1), "each scaled column should have mean 0 and std 1"
    ns = value("needs_scaling")
    if Ellipsis in ns.values():
        raise NotImplementedError
    assert ns == {"random_forest": False, "kmeans": True, "logistic_regression": True}, "trees split on thresholds (no scaling needed); k-means uses distances and logistic regression is regularised, so both need scaling"
TESTS['3.14'] = _test_3_14

def _test_3_15():
    y, p = [10, 10, 10, 10, 10], [8, 13, 9, 12, -10]
    expect("mae", (y, p), 5.6, tol=1e-9)
    expect("rmse", (y, p), math.sqrt((4 + 9 + 1 + 4 + 400) / 5), tol=1e-9)
    expect("rmse", ([1, 2], [1, 2]), 0.0, tol=1e-9)
TESTS['3.15'] = _test_3_15

def _test_3_16():
    f = need("bootstrap_diff_ci")
    lo, hi = value("ci")
    obs = group_a.mean() - group_b.mean()
    assert lo < obs < hi, f"the observed difference {obs:.2f} should lie inside your interval ({lo:.2f}, {hi:.2f})"
    assert 2 < hi - lo < 9, f"interval width {hi - lo:.2f} looks off for n=200 per group (expect roughly 5–6)"
    assert tuple(f(group_a, group_b)) == tuple(f(group_a, group_b)), "use the seed so results are repeatable"
    assert lo > 0, "with these groups the interval should sit above 0"
TESTS['3.16'] = _test_3_16

print("✅ Setup complete: 16 exercises ready. Work top to bottom.")

### Exercise 3.1 · Stratified train/test split
<small>📖 Card 3.1 in the visual guide</small>

Create `X` (all columns of `fleet` except `failed`) and `y` (`failed`).
Split into `X_tr, X_te, y_tr, y_te` with 25% test, **stratified** on y, `random_state=42`.

In [ ]:
X = ...
y = ...
X_tr, X_te, y_tr, y_te = ..., ..., ..., ...

In [ ]:
check("3.1")

<details><summary>💡 Show a solution</summary>

```python
X = fleet.drop(columns="failed")
y = fleet["failed"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
```

</details>

### Exercise 3.2 · A leak-free pipeline
<small>📖 Card 3.2 in the visual guide</small>

Build `model`: a `Pipeline` with
1. a `ColumnTransformer` that **median-imputes then scales** `["age_years", "avg_load_pct", "last_inspection_score"]` and **one-hot encodes** `["region", "manufacturer"]` (with `handle_unknown="ignore"`);
2. a `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42)`.

Fit it on the training data and create `proba` = predicted probability of failure for `X_te`.

In [ ]:
num = ["age_years", "avg_load_pct", "last_inspection_score"]
cat = ["region", "manufacturer"]
pre = ...
model = ...
proba = ...

In [ ]:
check("3.2")

<details><summary>💡 Show a solution</summary>

```python
num = ["age_years", "avg_load_pct", "last_inspection_score"]
cat = ["region", "manufacturer"]
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
model = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42)),
])
model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
```

</details>

### Exercise 3.3 · Metrics for rare events
<small>📖 Card 3.3 in the visual guide</small>

Using `y_te` and `proba`, compute:
- `base_rate`: the share of failures in `y_te`
- `roc`: ROC AUC, and `pr_auc`: average precision (PR-AUC)
- `precision`, `recall`: for the **failure class** when you flag assets with `proba >= 0.5`
- `accuracy_lazy`: the accuracy of a lazy model that predicts “no failure” for every asset

In [ ]:
base_rate = ...
roc = ...
pr_auc = ...
precision = ...
recall = ...
accuracy_lazy = ...

In [ ]:
check("3.3")

<details><summary>💡 Show a solution</summary>

```python
base_rate = y_te.mean()
roc = roc_auc_score(y_te, proba)
pr_auc = average_precision_score(y_te, proba)
pred = proba >= 0.5
precision = precision_score(y_te, pred)
recall = recall_score(y_te, pred)
accuracy_lazy = 1 - base_rate
```

</details>

### Exercise 3.4 · Precision@k and a business threshold
<small>📖 Card 3.4 in the visual guide</small>

**a)** Write `precision_at_k(y_true, scores, k)`: the share of true failures among the `k` highest scores.

**b)** `p_at_50 = precision_at_k(y_te, proba, 50)`

**c)** `thr_80`: the **highest** threshold that still catches at least 80% of failures (recall ≥ 0.8). Use `precision_recall_curve`.

<details><summary>🔎 Hint</summary>

`np.argsort(scores)[::-1][:k]` gives the positions of the top k. For (c): `prec, rec, thr = precision_recall_curve(y_te, proba)`; `rec[:-1]` lines up with `thr`.

</details>

In [ ]:
def precision_at_k(y_true, scores, k):
    ...

p_at_50 = ...
thr_80 = ...

In [ ]:
check("3.4")

<details><summary>💡 Show a solution</summary>

```python
def precision_at_k(y_true, scores, k):
    top = np.argsort(scores)[::-1][:k]
    return np.asarray(y_true)[top].mean()

p_at_50 = precision_at_k(y_te, proba, 50)
prec, rec, thr = precision_recall_curve(y_te, proba)
thr_80 = thr[rec[:-1] >= 0.8].max()
```

</details>

### Exercise 3.5 · Permutation importance
<small>📖 Card 3.5 in the visual guide</small>

Compute permutation importance of `model` on the **test** set (`scoring="average_precision"`, `n_repeats=5`, `random_state=42`).
Create `importances`: a Series indexed by column name, sorted from most to least important, and `top_feature`: its first index.

In [ ]:
importances = ...
top_feature = ...

In [ ]:
check("3.5")

<details><summary>💡 Show a solution</summary>

```python
pi = permutation_importance(model, X_te, y_te, scoring="average_precision", n_repeats=5, random_state=42)
importances = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
top_feature = importances.index[0]
```

</details>

### Exercise 3.6 · Cross-validation for time series
<small>📖 Card 3.6 in the visual guide</small>

Using `ts_X`, `ts_y` (already in time order), compute `cv_mae`: an array of the 5 fold MAEs from `cross_val_score` with
`Ridge()`, `TimeSeriesSplit(n_splits=5)` and `scoring="neg_mean_absolute_error"`. Make the numbers **positive**.

Then set `kfold_ok_for_time_series` to True or False: is a shuffled `KFold` OK here?

In [ ]:
cv_mae = ...
kfold_ok_for_time_series = ...

In [ ]:
check("3.6")

<details><summary>💡 Show a solution</summary>

```python
scores = cross_val_score(Ridge(), ts_X, ts_y, cv=TimeSeriesSplit(n_splits=5), scoring="neg_mean_absolute_error")
cv_mae = -scores
kfold_ok_for_time_series = False   # shuffling would let the model train on the future
```

</details>

### Exercise 3.7 · Forecast load and beat a naive baseline
<small>📖 Card 3.7 in the visual guide</small>

**a)** Write `make_features(d)`: returns a copy of `d` with columns `hour`, `dow`, `month`, `lag_24`, `lag_168`
(load 24 h and 168 h earlier) and `roll_24` (mean of the 24 hours ending 24 h ago). Drop rows with NaN.

**b)** `feat = make_features(load_ts)`. Use the **last 30 days** (720 hours) as test and everything before as train.
Fit `HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)` on all columns except `load`.

**c)** `mae_model`: MAE on the test period. `mae_naive`: MAE of the naive forecast “same hour last week” (`lag_168`).

In [ ]:
def make_features(d):
    ...

feat = ...
mae_model = ...
mae_naive = ...

In [ ]:
check("3.7")

<details><summary>💡 Show a solution</summary>

```python
def make_features(d):
    out = d.copy()
    out["hour"], out["dow"], out["month"] = out.index.hour, out.index.dayofweek, out.index.month
    for lag in [24, 168]:
        out[f"lag_{lag}"] = out["load"].shift(lag)
    out["roll_24"] = out["load"].shift(24).rolling(24).mean()
    return out.dropna()

feat = make_features(load_ts)
train, test = feat.iloc[:-720], feat.iloc[-720:]
cols = [c for c in feat.columns if c != "load"]
reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42).fit(train[cols], train["load"])
mae_model = mean_absolute_error(test["load"], reg.predict(test[cols]))
mae_naive = mean_absolute_error(test["load"], test["lag_168"])
```

</details>

### Exercise 3.8 · Isolation Forest
<small>📖 Card 3.8 in the visual guide</small>

Fit `IsolationForest(contamination=0.01, random_state=42)` on `sensors`. Create `flags` (a boolean array, True = anomaly)
and `n_flagged`. Is the last row (the planted anomaly) flagged?

In [ ]:
iso = ...
flags = ...
n_flagged = ...

In [ ]:
check("3.8")

<details><summary>💡 Show a solution</summary>

```python
iso = IsolationForest(contamination=0.01, random_state=42).fit(sensors)
flags = iso.predict(sensors) == -1
n_flagged = flags.sum()
```

</details>

### Exercise 3.9 · Grid search
<small>📖 Card 3.9 in the visual guide</small>

Run `GridSearchCV` on a pipeline of your `pre` + `RandomForestClassifier(class_weight="balanced", random_state=42)`
over `{"clf__n_estimators": [100, 200], "clf__max_depth": [5, None]}` with `cv=3`, `scoring="average_precision"`, `n_jobs=-1`, on the training data.
Store the fitted search in `grid` and its best settings in `best_params`.

In [ ]:
grid = ...
best_params = ...

In [ ]:
check("3.9")

<details><summary>💡 Show a solution</summary>

```python
grid = GridSearchCV(
    Pipeline([("pre", pre), ("clf", RandomForestClassifier(class_weight="balanced", random_state=42))]),
    {"clf__n_estimators": [100, 200], "clf__max_depth": [5, None]},
    cv=3, scoring="average_precision", n_jobs=-1,
)
grid.fit(X_tr, y_tr)
best_params = grid.best_params_
```

</details>

### Exercise 3.10 · Spot overfitting
<small>📖 Card 3.10 in the visual guide</small>

Split `x_over, y_over` with `test_size=0.3, random_state=0`. For each `max_depth` in `[1, 3, 5, 10, None]`, fit a
`DecisionTreeRegressor(max_depth=d, random_state=0)` and record train and test MAE.
Create `results` (a DataFrame with columns `depth`, `train_mae`, `test_mae`) and `best_depth` (the depth with the lowest test MAE).

In [ ]:
results = ...
best_depth = ...

In [ ]:
check("3.10")

<details><summary>💡 Show a solution</summary>

```python
xtr, xte, ytr, yte = train_test_split(x_over, y_over, test_size=0.3, random_state=0)
rows = []
for d in [1, 3, 5, 10, None]:
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(xtr, ytr)
    rows.append({"depth": d, "train_mae": mean_absolute_error(ytr, m.predict(xtr)),
                 "test_mae": mean_absolute_error(yte, m.predict(xte))})
results = pd.DataFrame(rows)
best_depth = results.loc[results["test_mae"].idxmin(), "depth"]
```

</details>

### Exercise 3.11 · From scratch: gradient descent and metrics
<small>📖 Card 3.11 in the visual guide</small>

**a)** `fit_linear_gd(X, y, lr=0.1, epochs=500)`: add an intercept column, start with zero weights, and repeat
`w -= lr * gradient_of_MSE`. Return the weights `[intercept, w1, w2, ...]`.

**b)** `confusion_metrics(y_true, y_pred)` returns `(precision, recall, f1)` computed by hand with NumPy. Return 0.0 instead of dividing by zero.

<details><summary>🔎 Hint</summary>

Gradient of MSE: `2 / n * Xb.T @ (Xb @ w - y)`. TP = `np.sum(y_true & y_pred)` for boolean arrays.

</details>

In [ ]:
def fit_linear_gd(X, y, lr=0.1, epochs=500):
    ...

def confusion_metrics(y_true, y_pred):
    ...

In [ ]:
check("3.11")

<details><summary>💡 Show a solution</summary>

```python
def fit_linear_gd(X, y, lr=0.1, epochs=500):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        grad = 2 / len(y) * Xb.T @ (Xb @ w - y)
        w -= lr * grad
    return w

def confusion_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, bool), np.asarray(y_pred, bool)
    tp = np.sum(y_true & y_pred)
    fp = np.sum(~y_true & y_pred)
    fn = np.sum(y_true & ~y_pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1
```

</details>

### Exercise 3.12 · Logistic regression on age
<small>📖 Card 3.12 in the visual guide</small>

Fit `logreg = LogisticRegression()` on `fleet[["age_years"]]` → `fleet["failed"]`. Then:
- `p_at_30`: the predicted probability of failure for a 30-year-old transformer
- `odds_per_year`: `exp(coefficient)`, how much each extra year multiplies the odds of failure

In [ ]:
logreg = ...
p_at_30 = ...
odds_per_year = ...

In [ ]:
check("3.12")

<details><summary>💡 Show a solution</summary>

```python
logreg = LogisticRegression().fit(fleet[["age_years"]], fleet["failed"])
p_at_30 = logreg.predict_proba(pd.DataFrame({"age_years": [30]}))[0, 1]
odds_per_year = np.exp(logreg.coef_[0, 0])
```

</details>

### Exercise 3.13 · k-means
<small>📖 Card 3.13 in the visual guide</small>

Fit `km = KMeans(n_clusters=3, n_init=10, random_state=0)` on `points`. Create `sizes`: the number of points in each cluster, as a **sorted list**.

In [ ]:
km = ...
sizes = ...

In [ ]:
check("3.13")

<details><summary>💡 Show a solution</summary>

```python
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(points)
sizes = sorted(np.bincount(km.labels_).tolist())
```

</details>

### Exercise 3.14 · Feature scaling
<small>📖 Card 3.14 in the visual guide</small>

Take `X_small = fleet[["age_years", "avg_load_pct"]].head(100)` and create `scaled` with `StandardScaler`.
Then fill in `needs_scaling`: a dict with True/False for `"random_forest"`, `"kmeans"`, `"logistic_regression"`.

In [ ]:
X_small = fleet[["age_years", "avg_load_pct"]].head(100)
scaled = ...
needs_scaling = {"random_forest": ..., "kmeans": ..., "logistic_regression": ...}

In [ ]:
check("3.14")

<details><summary>💡 Show a solution</summary>

```python
X_small = fleet[["age_years", "avg_load_pct"]].head(100)
scaled = StandardScaler().fit_transform(X_small)
needs_scaling = {"random_forest": False, "kmeans": True, "logistic_regression": True}
```

</details>

### Exercise 3.15 · MAE and RMSE by hand
<small>📖 Card 3.15 in the visual guide</small>

Write `mae(y, pred)` and `rmse(y, pred)` using NumPy only.

In [ ]:
def mae(y, pred):
    ...

def rmse(y, pred):
    ...

In [ ]:
check("3.15")

<details><summary>💡 Show a solution</summary>

```python
def mae(y, pred):
    return np.mean(np.abs(np.asarray(y) - np.asarray(pred)))

def rmse(y, pred):
    return np.sqrt(np.mean((np.asarray(y) - np.asarray(pred)) ** 2))
```

</details>

### Exercise 3.16 · Bootstrap confidence interval
<small>📖 Card 3.16 in the visual guide</small>

Write `bootstrap_diff_ci(a, b, n_boot=2000, seed=0)`: resample `a` and `b` **with replacement** `n_boot` times
(use `np.random.default_rng(seed)`), record the difference in means each time, and return the 2.5th and 97.5th
percentiles as a tuple `(lo, hi)`. Then `ci = bootstrap_diff_ci(group_a, group_b)`.

In [ ]:
def bootstrap_diff_ci(a, b, n_boot=2000, seed=0):
    ...

ci = ...

In [ ]:
check("3.16")

<details><summary>💡 Show a solution</summary>

```python
def bootstrap_diff_ci(a, b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    diffs = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return lo, hi

ci = bootstrap_diff_ci(group_a, group_b)
```

</details>

### 🗣 Say it out loud · 3.17

**“Your failure model looks good in the notebook. How would you put it into production on Azure, and keep it trustworthy?”**

*Answer out loud first, or type your answer in the cell below. Then compare.*

✍️ *Your answer (double-click to edit):*



<details><summary>💬 Show a model answer</summary>

Data from SCADA, work orders and weather lands in a Fabric or Synapse lakehouse. I'd train in an Azure ML workspace and track every run in MLflow: parameters, metrics like PR-AUC and precision@k, and the model itself, then register the version I'd ship. For a monthly risk list, a batch endpoint is enough; if an app needs scores in real time, an online endpoint. Retraining runs on a schedule as an Azure ML pipeline. I monitor data drift on the inputs, and the model's real precision once new failures are recorded. Results go into Power BI for the crews. Each model gets a model card covering its data, metrics, limits and owner, which is what a governance or regulator review needs.

</details>

## 🏁 Your progress
Run the cell below to see which exercises you've passed.

In [ ]:
progress()